In [0]:
%pip install yfinance

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta

TICKERS = ["AAPL", "MSFT", "GOOGL", "AMZN", "META", "NVDA", "TSLA", "NFLX", "AMD", "INTC"]

end_date = datetime.today()
start_date = end_date - timedelta(days=365)

all_data = []

for ticker in TICKERS:
    print(f"Downloading: {ticker}")
    stock = yf.Ticker(ticker)
    df = stock.history(start=start_date, end=end_date)
    
    if df.empty:
        print(f"  No data for {ticker}")
        continue
    
    df = df.reset_index()
    df["ticker"] = ticker
    df = df.rename(columns={
        "Date": "date",
        "Open": "open",
        "High": "high",
        "Low": "low",
        "Close": "close",
        "Volume": "volume",
    })
    df = df[["ticker", "date", "open", "high", "low", "close", "volume"]]
    all_data.append(df)
    print(f"  Done: {len(df)} rows")

pandas_df = pd.concat(all_data, ignore_index=True)
print(f"\nTotal rows: {len(pandas_df)}")

Downloading: AAPL
  Done: 250 rows
Downloading: MSFT
  Done: 250 rows
Downloading: GOOGL
  Done: 250 rows
Downloading: AMZN
  Done: 250 rows
Downloading: META
  Done: 250 rows
Downloading: NVDA
  Done: 250 rows
Downloading: TSLA
  Done: 250 rows
Downloading: NFLX
  Done: 250 rows
Downloading: AMD
  Done: 250 rows
Downloading: INTC
  Done: 250 rows

Total rows: 2500


In [0]:
spark_df = spark.createDataFrame(pandas_df)

spark_df.show(10)

spark_df.printSchema()

+------+-------------------+------------------+------------------+------------------+------------------+---------+
|ticker|               date|              open|              high|               low|             close|   volume|
+------+-------------------+------------------+------------------+------------------+------------------+---------+
|  AAPL|2025-04-09 04:00:00|171.20381523449285| 199.7394480269994|171.14407803694525|197.98709106445312|184395900|
|  AAPL|2025-04-10 04:00:00|188.24952597387417| 193.9347385570105|  182.205859833413|189.59365844726562|121880000|
|  AAPL|2025-04-11 04:00:00|185.29243322025624|198.67409816298056|185.25259829067514|197.29013061523438| 87435900|
|  AAPL|2025-04-14 04:00:00|210.52246440005902|212.01595519155484|200.28707539108044|201.64117431640625|101352900|
|  AAPL|2025-04-15 04:00:00|200.98401841430368|202.62685207462172|198.93296032454862|201.26280212402344| 51343900|
|  AAPL|2025-04-16 04:00:00|197.49920369317184| 199.8290454554037|191.5351922423

In [0]:
spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bigtech_bronze_stocks")

print("Bronze layer saved as table: bigtech_bronze_stocks")

bronze_check = spark.table("bigtech_bronze_stocks")
print(f"Rows in Bronze: {bronze_check.count()}")

Bronze layer saved as table: bigtech_bronze_stocks
Rows in Bronze: 2500


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType

bronze_df = spark.table("bigtech_bronze_stocks")

print(f"Bronze rows: {bronze_df.count()}")

silver_df = bronze_df.withColumn("date", F.to_date(F.col("date")))

silver_df = silver_df.dropDuplicates(["ticker", "date"])

silver_df = silver_df.dropna(subset=["open", "high", "low", "close"])

silver_df = silver_df.fillna({"volume": 0})

silver_df = silver_df \
    .withColumn("open", F.round(F.col("open"), 2)) \
    .withColumn("high", F.round(F.col("high"), 2)) \
    .withColumn("low", F.round(F.col("low"), 2)) \
    .withColumn("close", F.round(F.col("close"), 2))

silver_df = silver_df.orderBy("ticker", "date")

print(f"Silver rows: {silver_df.count()}")
silver_df.show(10)

Bronze rows: 2500
Silver rows: 2500
+------+----------+------+------+------+------+---------+
|ticker|      date|  open|  high|   low| close|   volume|
+------+----------+------+------+------+------+---------+
|  AAPL|2025-04-09| 171.2|199.74|171.14|197.99|184395900|
|  AAPL|2025-04-10|188.25|193.93|182.21|189.59|121880000|
|  AAPL|2025-04-11|185.29|198.67|185.25|197.29| 87435900|
|  AAPL|2025-04-14|210.52|212.02|200.29|201.64|101352900|
|  AAPL|2025-04-15|200.98|202.63|198.93|201.26| 51343900|
|  AAPL|2025-04-16| 197.5|199.83|191.54|193.43| 59732400|
|  AAPL|2025-04-17|196.34|197.97|193.58|196.13| 52164700|
|  AAPL|2025-04-21|192.43|192.96|188.99|192.32| 46742500|
|  AAPL|2025-04-22|195.27|200.72|195.12|198.87| 52976400|
|  AAPL|2025-04-23|205.11| 207.1|201.92|203.71| 52929200|
+------+----------+------+------+------+------+---------+
only showing top 10 rows


In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bigtech_silver_stocks")

print("Silver layer saved as table: bigtech_silver_stocks")
print(f"Rows: {spark.table('bigtech_silver_stocks').count()}")

Silver layer saved as table: bigtech_silver_stocks
Rows: 2500


In [0]:
from pyspark.sql.window import Window

silver = spark.table("bigtech_silver_stocks")

ticker_window = Window.partitionBy("ticker").orderBy("date")

window_7d = ticker_window.rowsBetween(-6, 0)
window_30d = ticker_window.rowsBetween(-29, 0)

prev_close = F.lag("close", 1).over(ticker_window)
gold_df = silver.withColumn(
    "daily_return_pct",
    F.round(((F.col("close") - prev_close) / prev_close) * 100, 4)
)

gold_df = gold_df \
    .withColumn("ma_7d", F.round(F.avg("close").over(window_7d), 2)) \
    .withColumn("ma_30d", F.round(F.avg("close").over(window_30d), 2))

gold_df = gold_df.withColumn(
    "volatility_30d",
    F.round(F.stddev("daily_return_pct").over(window_30d), 4)
)

gold_df = gold_df.withColumn(
    "daily_range_pct",
    F.round(((F.col("high") - F.col("low")) / F.col("close")) * 100, 4)
)

gold_df = gold_df.withColumn(
    "above_ma30",
    F.when(F.col("close") > F.col("ma_30d"), 1).otherwise(0)
)

print(f"Gold daily metrics: {gold_df.count()} rows")
gold_df.show(10)

Gold daily metrics: 2500 rows
+------+----------+------+------+------+------+---------+----------------+------+------+--------------+---------------+----------+
|ticker|      date|  open|  high|   low| close|   volume|daily_return_pct| ma_7d|ma_30d|volatility_30d|daily_range_pct|above_ma30|
+------+----------+------+------+------+------+---------+----------------+------+------+--------------+---------------+----------+
|  AAPL|2025-04-09| 171.2|199.74|171.14|197.99|184395900|            NULL|197.99|197.99|          NULL|        14.4452|         0|
|  AAPL|2025-04-10|188.25|193.93|182.21|189.59|121880000|         -4.2426|193.79|193.79|          NULL|         6.1818|         0|
|  AAPL|2025-04-11|185.29|198.67|185.25|197.29| 87435900|          4.0614|194.96|194.96|        5.8718|         6.8022|         1|
|  AAPL|2025-04-14|210.52|212.02|200.29|201.64|101352900|          2.2049|196.63|196.63|        4.3584|         5.8173|         1|
|  AAPL|2025-04-15|200.98|202.63|198.93|201.26| 51343

In [0]:
gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bigtech_gold_daily_metrics")

print("Gold daily metrics saved!")

company_summary = gold_df.groupBy("ticker").agg(
    F.round(F.last("close"), 2).alias("latest_close"),
    F.round(F.avg("close"), 2).alias("avg_close"),
    F.round(F.max("close"), 2).alias("max_close"),
    F.round(F.min("close"), 2).alias("min_close"),
    F.round(F.avg("volume"), 0).alias("avg_volume"),
    F.round(F.avg("daily_return_pct"), 4).alias("avg_daily_return"),
    F.round(F.max("daily_return_pct"), 4).alias("max_daily_return"),
    F.round(F.min("daily_return_pct"), 4).alias("min_daily_return"),
    F.round(F.avg("volatility_30d"), 4).alias("avg_volatility"),
    F.sum("above_ma30").alias("days_above_ma30"),
    F.count("close").alias("total_trading_days"),
)

company_summary = company_summary.withColumn(
    "pct_days_above_ma30",
    F.round((F.col("days_above_ma30") / F.col("total_trading_days")) * 100, 2)
)

company_summary = company_summary.orderBy(F.desc("avg_volatility"))

company_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bigtech_gold_company_summary")

print("Gold company summary saved!")
company_summary.show()

Gold daily metrics saved!
Gold company summary saved!
+------+------------+---------+---------+---------+------------+----------------+----------------+----------------+--------------+---------------+------------------+-------------------+
|ticker|latest_close|avg_close|max_close|min_close|  avg_volume|avg_daily_return|max_daily_return|min_daily_return|avg_volatility|days_above_ma30|total_trading_days|pct_days_above_ma30|
+------+------------+---------+---------+---------+------------+----------------+----------------+----------------+--------------+---------------+------------------+-------------------+
|  INTC|       58.95|    32.74|    58.95|    18.84| 1.0118276E8|          0.4852|         22.7711|        -17.0287|        3.8556|            158|               250|               63.2|
|   AMD|      231.82|   180.12|   264.33|    85.56| 4.7771728E7|          0.4191|          23.708|        -17.3144|        3.6648|            147|               250|               58.8|
|  TSLA|      34

In [0]:
print("=== TOP 5 NAJBARDZIEJ ZMIENNYCH SPÓŁEK ===")
spark.sql("""
    SELECT 
        ticker,
        latest_close,
        avg_volatility,
        pct_days_above_ma30
    FROM bigtech_gold_company_summary
    ORDER BY avg_volatility DESC
    LIMIT 5
""").show()

=== TOP 5 NAJBARDZIEJ ZMIENNYCH SPÓŁEK ===
+------+------------+--------------+-------------------+
|ticker|latest_close|avg_volatility|pct_days_above_ma30|
+------+------------+--------------+-------------------+
|  INTC|       58.95|        3.8556|               63.2|
|   AMD|      231.82|        3.6648|               58.8|
|  TSLA|      343.25|        3.1149|               52.4|
|  NVDA|      182.08|        2.2804|               63.2|
|  META|      612.42|         2.174|               52.0|
+------+------------+--------------+-------------------+



In [0]:
print("=== NAJLEPSZY I NAJGORSZY DZIEŃ KAŻDEJ SPÓŁKI ===")
spark.sql("""
    SELECT 
        ticker,
        max_daily_return AS best_day_pct,
        min_daily_return AS worst_day_pct,
        ROUND(max_daily_return - min_daily_return, 2) AS spread
    FROM bigtech_gold_company_summary
    ORDER BY spread DESC
""").show()

=== NAJLEPSZY I NAJGORSZY DZIEŃ KAŻDEJ SPÓŁKI ===
+------+------------+-------------+------+
|ticker|best_day_pct|worst_day_pct|spread|
+------+------------+-------------+------+
|   AMD|      23.708|     -17.3144| 41.02|
|  INTC|     22.7711|     -17.0287|  39.8|
|  TSLA|      9.8031|     -14.2599| 24.06|
|  NFLX|     13.7723|     -10.0693| 23.84|
|  META|     11.2532|     -11.3338| 22.59|
|  AMZN|      9.5845|      -8.2696| 17.85|
|  MSFT|       7.625|      -9.9931| 17.62|
| GOOGL|      9.1383|      -7.2601|  16.4|
|  NVDA|      7.8722|      -6.8646| 14.74|
|  AAPL|      6.3136|      -4.9982| 11.31|
+------+------------+-------------+------+



In [0]:
print("=== EKSTREMALNE DNI (zmiana > 5% lub < -5%) ===")
spark.sql("""
    SELECT 
        ticker,
        date,
        close,
        daily_return_pct
    FROM bigtech_gold_daily_metrics
    WHERE ABS(daily_return_pct) > 5
    ORDER BY ABS(daily_return_pct) DESC
    LIMIT 10
""").show()

=== EKSTREMALNE DNI (zmiana > 5% lub < -5%) ===
+------+----------+------+----------------+
|ticker|      date| close|daily_return_pct|
+------+----------+------+----------------+
|   AMD|2025-10-06|203.71|          23.708|
|  INTC|2025-09-18| 30.57|         22.7711|
|   AMD|2026-02-04|200.19|        -17.3144|
|  INTC|2026-01-23| 45.07|        -17.0287|
|  TSLA|2025-06-05| 284.7|        -14.2599|
|  NFLX|2026-02-27| 96.24|         13.7723|
|  INTC|2026-01-21| 54.25|         11.7175|
|  INTC|2026-04-08| 58.95|         11.4156|
|   AMD|2025-10-08|235.56|         11.3706|
|  META|2025-10-30|665.36|        -11.3338|
+------+----------+------+----------------+



In [0]:
print("=== MIESIĄCE Z WYSOKĄ ŚREDNIĄ ZMIENNOŚCIĄ DZIENNĄ ===")
spark.sql("""
    SELECT
        ticker,
        YEAR(date) AS year,
        MONTH(date) AS month,
        COUNT(*) AS trading_days,
        ROUND(AVG(daily_return_pct), 4) AS avg_return,
        ROUND(AVG(volatility_30d), 4) AS avg_volatility,
        ROUND(MAX(daily_return_pct), 2) AS best_day,
        ROUND(MIN(daily_return_pct), 2) AS worst_day,
        ROUND(SUM(volume) / 1e9, 2) AS vol_billions
    FROM bigtech_gold_daily_metrics
    WHERE daily_return_pct IS NOT NULL
    GROUP BY ticker, YEAR(date), MONTH(date)
    HAVING AVG(volatility_30d) > 3
    ORDER BY avg_volatility DESC
""").show(20)

=== MIESIĄCE Z WYSOKĄ ŚREDNIĄ ZMIENNOŚCIĄ DZIENNĄ ===
+------+----+-----+------------+----------+--------------+--------+---------+------------+
|ticker|year|month|trading_days|avg_return|avg_volatility|best_day|worst_day|vol_billions|
+------+----+-----+------------+----------+--------------+--------+---------+------------+
|  INTC|2026|    2|          19|   -0.0547|         5.672|    5.71|    -6.19|        1.72|
|   AMD|2025|    4|          14|    0.1148|        5.3736|     5.3|    -8.41|        0.54|
|   AMD|2025|   11|          19|    -0.763|        5.1478|     9.0|    -7.84|        1.05|
|  INTC|2025|   10|          23|    0.8068|        5.0046|    7.12|    -4.27|        2.78|
|   AMD|2025|   10|          23|    2.1938|        4.9437|   23.71|    -7.72|        1.76|
|   AMD|2026|    2|          19|   -0.7318|        4.4632|    8.77|   -17.31|        0.82|
|  INTC|2026|    1|          20|    1.3866|        4.3235|   11.72|   -17.03|        2.95|
|  INTC|2026|    4|           5|    